## Send API requests

This notebook takes the shapefile of fire we want to validate, prepares all the API requests, sends them off and creates a log file of all sent jobs.

In [5]:
import pandas as pd
import geopandas as gpd
import fiona
import numpy as np
import requests
from src import config
from src import APIrequests

In [6]:
calfire_validation = gpd.read_file(config.PATH_FIRES_VALIDATION)
calfire_validation.head()
fire_names = calfire_validation['FIRE_NAME'].unique()
results = []

### Test if it works

In [7]:
fire_name = 'YORK'
print(fire_name)
bbox = APIrequests.create_bbox(fire_name, calfire_validation)
date_of_fire = APIrequests.get_fire_date(fire_name, calfire_validation)
query_data = APIrequests.create_query(fire_name, bbox.values[0], date_of_fire, 15)
try:
    # Use the 'json' parameter: it automatically sets 'Content-Type: application/json'
    # and runs json.dumps() for you.
    response = requests.post(config.URL_API, json=query_data)

    # 4. Check the results
    if response.status_code == 200 or response.status_code == 201:
        print("Success!")
        print(response.json()) # This is the data the API sends back
    else:
        print(f"Failed with status code: {response.status_code}")
        print(response.text) # This shows the error message from the API

except requests.exceptions.RequestException as e:
    print(f"A connection error occurred: {e}")

YORK
getting fire date
Success!
{'fire_event_name': 'YORK_date2023-07-28_range15_modealarm', 'status': 'Processing started', 'job_id': 'd5e9468d-3baf-4f46-8421-b51f4c271446'}


### Send off all jobs

In [ ]:

# Loop through each fire, date mode, sensor, and post-fire day range
for fire_name in fire_names:
    bbox = APIrequests.create_bbox(fire_name, calfire_validation)

    for post_fire_reference_point in config.POST_FIRE_REFERENCE_POINT:
        if post_fire_reference_point == 'alarm':
            date_of_fire = APIrequests.get_fire_date(fire_name, calfire_validation)
        else:
            date_of_fire = APIrequests.get_cont_date(fire_name, calfire_validation)

        if date_of_fire is None:
            for sensor in config.SENSORS:
                for post_fire_period in config.POST_FIRE_PERIOD:
                    APIrequests.append_result(results, fire_name, post_fire_reference_point, post_fire_period, sensor, 'skipped_no_cont_date')
            print(f"- {fire_name} ({post_fire_reference_point}): skipped — no CONT_DATE")
            continue

        for sensor in config.SENSORS:
            print(f"\n=== {fire_name} ({post_fire_reference_point}) — sensor: {sensor} ===")
            for post_fire_period in config.POST_FIRE_PERIOD:
                query_data = APIrequests.create_query(fire_name, bbox.values[0], date_of_fire, post_fire_period, post_fire_reference_point, sensor)

                try:
                    response = requests.post(config.URL_API, json=query_data)

                    if response.status_code == 200 or response.status_code == 201:
                        response_data = response.json()
                        APIrequests.append_result(results, fire_name, post_fire_reference_point, post_fire_period, sensor, 'success',
                                   fire_event_name=response_data.get('fire_event_name'),
                                   job_id=response_data.get('job_id'))
                        print(f"✓ [{sensor}] {fire_name} ({post_fire_reference_point}, {post_fire_period} days): {response_data.get('job_id')}")
                    else:
                        APIrequests.append_result(results, fire_name, post_fire_reference_point, post_fire_period, sensor, f'failed_{response.status_code}',
                                    fire_event_name=query_data['fire_event_name'])
                        print(f"✗ [{sensor}] {fire_name} ({post_fire_reference_point}, {post_fire_period} days): Failed with {response.status_code}")

                except requests.exceptions.RequestException as e:
                    APIrequests.append_result(results, fire_name, post_fire_reference_point, post_fire_period, sensor, 'error',
                                    fire_event_name=query_data['fire_event_name'])
                    print(f"✗ [{sensor}] {fire_name} ({post_fire_reference_point}, {post_fire_period} days): Connection error")

# Create dataframe and save to CSV
df_results = pd.DataFrame(results)
df_results.to_csv(config.PATH_JOBS_LOG, index=False)

print(f"\nProcessed {len(results)} requests. Results saved to {config.PATH_JOBS_LOG}")
print(df_results['sensor'].value_counts())
display(df_results)


### Check status

In [ ]:
fires = pd.read_csv(config.PATH_JOBS_LOG)

# Collect status for each row
statuses = []

for idx, row in fires.iterrows():
    if row['status'] != 'success':
        statuses.append(999)
        continue

    request = requests.get(f"{config.URL_RESULT}/{row['fire_event_name']}/{row['job_id']}")
    status = request.json().get('status')
    print(f"Job {row['fire_event_name']}: {status}")
    statuses.append(status)

# Add status column to dataframe
fires['job_status'] = statuses

# Save updated dataframe
fires.to_csv(config.PATH_JOBS_LOG, index=False)

print(f"\nUpdated {config.PATH_JOBS_LOG} with job_status column")
fires.head()
